# G.654.E Paper-Reference Validation Report
## `EGN_adaptive.py` + `run_G654.py`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kimheeseo/LSCNS/blob/main/paper/corningG654EIWCS/G654E_EGN_adaptive_validation_report.ipynb)

목적: 논문/발표의 출력 SNR에 맞추는 보정계수 없이, 공개된 G.654.E fiber/system parameter를 `EGN_adaptive.py`에 입력했을 때 paper-reference launch-power/SNR 경향이 재현되는지 검증합니다.

> `Paper reference`는 기존 repository notebook의 **figure-based fitted equation**입니다. 원 논문에 인쇄된 폐형식 수식이라고 주장하지 않습니다.

In [1]:
from pathlib import Path
module_path = Path("EGN_adaptive.py")
if not module_path.exists():
    import requests
    url = "https://raw.githubusercontent.com/kimheeseo/LSCNS/main/paper/corningG654EIWCS/EGN_adaptive.py"
    r = requests.get(url, timeout=30); r.raise_for_status()
    module_path.write_text(r.text, encoding="utf-8")
print("EGN_adaptive.py ready")

EGN_adaptive.py ready


In [2]:
import math, numpy as np, pandas as pd, matplotlib.pyplot as plt
from EGN_adaptive import WDMSystem, Span, GNIntegralOptions, launch_power_vs_gsnr

## 1. Input comparison

| Parameter | Paper / presentation | `run_G654.py` | Status |
|---|---:|---:|---|
| Fiber | G.654.E | G.654.E | Same |
| WDM channels | 90 | 90 | Same |
| Modulation | 64QAM | 64QAM | Same |
| Symbol rate | 95 GBd | 95 GBd | Same |
| Channel spacing | Nyquist spaced | 95 GHz | Same interpretation |
| TRX SNR | 18 dB | 18 dB | Same |
| Span length | 80 km | 80 km | Same |
| Number of spans | 1–50 stated | **30 fixed** | **Different** |
| EDFA NF | 5 dB | 5 dB | Same |
| Total gain bandwidth | 4.8 THz | **Not explicitly limited** | **Different** |
| Shannon gap | 3 dB | 3 dB | Same (capacity only) |
| Attenuation | 0.166 dB/km | 0.166 dB/km | Same |
| Effective area | 125 μm² | 125 μm² | Same |
| Dispersion | 21 ps/(nm·km) | 21 ps/(nm·km) | Same |
| n2 | 2.2×10⁻²⁰ m²/W | 2.2×10⁻²⁰ m²/W | Same |

90×95 GHz = **8.55 THz** occupied comb. 따라서 4.8-THz gain-bandwidth condition은 현재 코드와 동일하다고 간주하지 않습니다.

In [3]:
alpha_db_per_km=0.166; Aeff_um2=125.0; D_ps_nm_km=21.0; n2=2.2e-20; wavelength_nm=1550.0
gamma_W_inv_km=2*math.pi*n2/((wavelength_nm*1e-9)*(Aeff_um2*1e-12))*1e3
print(f"gamma = {gamma_W_inv_km:.9f} 1/(W km)")

gamma = 0.713445557 1/(W km)


In [4]:
system=WDMSystem.equispaced(90,95.0,95.0,0.0,pulse_shape="rect",rolloff=0.0)
cut_index=45
span=Span(80.0,0.166,gamma_W_inv_km,D_ps_nm_km=21.0,wavelength_nm=1550.0,noise_figure_db=5.0)
spans=[span]*30
opt=GNIntegralOptions(sobol_power=14,seed=1,accumulation="coherent",z_quadrature_order=64,egn_frequency_order=64)
pgrid=np.arange(-10.0,11.0,1.0)
result=launch_power_vs_gsnr(system,spans,cut_index,pgrid,modulation="64QAM",trx_snr_db=18.0,shannon_gap_db=3.0,gn_options=opt,receiver_points=3,nli_model="egn_sci",use_cubic_scaling=True)
print("Calculation completed: 21 launch-power points")

Calculation completed: 21 launch-power points


## 2. Paper-reference equation

기존 reproduction notebook의 figure-fit:

$$SNR_{paper}(P)=10\log_{10}\left[\frac{1}{0.02254\,10^{-P/10}+0.000817\,10^{P/5}+0.0158}\right]$$

이 계수들은 **비교 단계에서만** 사용하며 `EGN_adaptive.py`의 physics calculation에는 입력하지 않습니다.

In [5]:
paper=10*np.log10(1/(0.02254*10**(-pgrid/10)+0.000817*10**(pgrid/5)+0.0158))
code_snr=np.asarray(result["gsnr_db"])
df=pd.DataFrame({"P_dBm":pgrid,"Paper_dB":paper,"Code_dB":code_snr})
df["AbsErr_dB"]=np.abs(df.Code_dB-df.Paper_dB)
df["RelErr_pct"]=100*df.AbsErr_dB/np.abs(df.Paper_dB)
print(df.round(4).to_string(index=False))

 P_dBm  Paper_dB  Code_dB  AbsErr_dB  RelErr_pct
  -10.0    6.1761   6.0188     0.1573      2.5472
   -9.0    7.1029   6.9480     0.1548      2.1801
   -8.0    8.0124   7.8606     0.1518      1.8944
   -7.0    8.9008   8.7529     0.1480      1.6623
   -6.0    9.7640   9.6209     0.1431      1.4657
   -5.0   10.5969  10.4600     0.1369      1.2918
   -4.0   11.3938  11.2651     0.1287      1.1291
   -3.0   12.1482  12.0309     0.1173      0.9659
   -2.0   12.8526  12.7515     0.1011      0.7869
   -1.0   13.4977  13.4209     0.0769      0.5695
    0.0   14.0719  14.0327     0.0392      0.2786
    1.0   14.5594  14.5801     0.0206      0.1417
    2.0   14.9385  15.0548     0.1163      0.7786
    3.0   15.1785  15.4461     0.2676      1.7630
    4.0   15.2392  15.7390     0.4999      3.2801
    5.0   15.0727  15.9119     0.8392      5.5677
    6.0   14.6331  15.9346     1.3015      8.8939
    7.0   13.8913  15.7686     1.8773     13.5143
    8.0   12.8485  15.3724     2.5239     19.6436
 

In [6]:
err=code_snr-paper; ae=np.abs(err)
for label,mask in [("-10 to +4 dBm",pgrid<=4),("-10 to +10 dBm",np.ones_like(pgrid,dtype=bool))]:
    mae=ae[mask].mean(); rmse=np.sqrt(np.mean(err[mask]**2)); mape=np.mean(ae[mask]/np.abs(paper[mask]))*100; mx=ae[mask].max()
    print(f"{label}: MAE={mae:.4f} dB, RMSE={rmse:.4f} dB, MAPE={mape:.4f}%, Max={mx:.4f} dB")
i4=np.where(pgrid==4)[0][0]
print(f"+4 dBm: Paper={paper[i4]:.4f} dB, Code={code_snr[i4]:.4f} dB, Delta={err[i4]:+.4f} dB")
print(f"Paper grid optimum: {pgrid[np.argmax(paper)]:+.0f} dBm, {paper.max():.4f} dB")
print(f"Code grid optimum:  {pgrid[np.argmax(code_snr)]:+.0f} dBm, {code_snr.max():.4f} dB")

-10 to +4 dBm: MAE=0.1506 dB, RMSE=0.1855 dB, MAPE=1.3823%, Max=0.4999 dB
-10 to +10 dBm: MAE=0.7494 dB, RMSE=1.3279 dB, MAPE=6.3540%, Max=3.7610 dB
+4 dBm: Paper=15.2392 dB, Code=15.7390 dB, Delta=+0.4999 dB
Paper grid optimum: +4 dBm, 15.2392 dB
Code grid optimum:  +6 dBm, 15.9346 dB


In [7]:
plt.figure(figsize=(9,5.5))
plt.plot(pgrid,paper,marker="s",label="Paper reference (figure-fitted equation)")
plt.plot(pgrid,code_snr,marker="o",label="EGN_adaptive.py + run_G654.py")
plt.axvline(4,linestyle="--",linewidth=1,label="+4 dBm reference")
plt.xlabel("Launch power per channel (dBm)"); plt.ylabel("SNR_tot (dB)")
plt.title("G.654.E: Paper Reference vs EGN_adaptive.py"); plt.grid(True,alpha=.3); plt.legend(); plt.tight_layout()
plt.savefig("g654e_paper_vs_code.svg",bbox_inches="tight"); plt.show()
print("Figure generated: g654e_paper_vs_code.svg")

Figure generated: g654e_paper_vs_code.svg


![G.654.E Paper Reference vs EGN_adaptive.py](./g654e_paper_vs_code.svg)

## 3. Interpretation and conclusion

- Paper output curve의 계수는 EGN 계산에 사용하지 않았습니다. 즉 **output-target fitting이 아닙니다**.
- Paper와 같은 핵심 fiber/transceiver input을 사용한 독립 ASE + NLI + TRX-noise 계산에서 **-10~+4 dBm 구간 MAE ≈ 0.151 dB, MAPE ≈ 1.38%**의 일치를 보입니다.
- 이는 코드의 구현 타당성과 low-to-moderate launch-power 영역 재현성을 강하게 지지합니다.
- 반면 +5 dBm 이상에서는 차이가 커지므로 전체 범위를 동일하다고 주장하지 않습니다.
- 남은 차이는 30-span 고정, 4.8-THz bandwidth 미적용, 90-channel 때문에 strict `egn_full` 대신 `egn_sci` compatibility path를 사용한 점 등이 후보입니다.

> **Conclusion:** `EGN_adaptive.py` reproduces the paper-reference G.654.E launch-power/SNR trend without output-target fitting. Close agreement in the low-to-moderate launch-power region supports the physical validity and reproducibility of the implementation, while high-power deviation is retained transparently rather than removed by empirical tuning.